# Rohil Utture — DATA 266 Homework 1
## Task 3: CUDA C Matrix Multiplication + Profiling

**Run on a GPU runtime:** Runtime → Change runtime type → Hardware accelerator = **GPU** → Save.

Parameters: `SEED = 2708` (seeds the random input matrices in `matmul.cu`).

### 0. Confirm GPU and auto-select the NVCC `-arch` (matches Week-1 demo)

In [1]:
import os, torch
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> GPU.'
name = torch.cuda.get_device_name(0)
cap  = torch.cuda.get_device_capability(0)      # (major, minor)
os.environ['SM'] = f'sm_{cap[0]}{cap[1]}'        # e.g. sm_75 (T4), sm_80 (A100)
print('GPU:', name, '| compute capability:', cap, '| NVCC arch:', os.environ['SM'])
!nvidia-smi

GPU: Tesla T4 | compute capability: (7, 5) | NVCC arch: sm_75
Mon Aug 31 00:24:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |          

### 1. Write the CUDA source
One thread computes one output element `C[row][col]`. Threads form **16×16 = 256-thread blocks**; blocks form a 2-D **grid** of `ceil(N/16) × ceil(N/16)` that tiles the whole `N×N` output. A `row<N && col<N` guard handles sizes that aren't multiples of 16. Full explanation is in the source header.

In [2]:
%%writefile matmul.cu
// matmul.cu — DATA 266 HW1, Task 3 (Rohil, SID4=2708)
// Square matrix multiply C = A * B on CPU (baseline) and GPU (CUDA),
// timing CPU compute, GPU kernel, and H2D+D2H transfers for N = 256, 1024, 4096.
//
// Build (Colab GPU):   !nvcc -O3 -arch=sm_75 matmul.cu -o matmul
// Run:                 !./matmul
//
// -------------------- BLOCKS AND THREADS --------------------
// The output matrix C is N x N. We map ONE CUDA THREAD to ONE output element C[row][col].
// Threads are grouped into 2-D BLOCKS of BLOCK x BLOCK threads (here 16x16 = 256 threads/block).
// Blocks are arranged in a 2-D GRID that tiles the whole N x N output:
//     grid.x = ceil(N / BLOCK)   covers the columns
//     grid.y = ceil(N / BLOCK)   covers the rows
// Inside the kernel each thread recovers its global coordinates from its block and thread indices:
//     col = blockIdx.x * blockDim.x + threadIdx.x
//     row = blockIdx.y * blockDim.y + threadIdx.y
// Because N may not divide evenly by BLOCK, we launch a few extra threads and guard with
// `if (row < N && col < N)` so out-of-range threads do nothing.
// ------------------------------------------------------------

#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <chrono>
#include <cuda_runtime.h>

#define BLOCK 16   // 16 x 16 = 256 threads per block

#define CUDA_CHECK(call)                                                        \
    do {                                                                        \
        cudaError_t err = (call);                                               \
        if (err != cudaSuccess) {                                               \
            fprintf(stderr, "CUDA error %s at %s:%d\n",                         \
                    cudaGetErrorString(err), __FILE__, __LINE__);              \
            exit(EXIT_FAILURE);                                                 \
        }                                                                       \
    } while (0)

// GPU kernel: each thread computes one element of C.
__global__ void matmulKernel(const float* A, const float* B, float* C, int N) {
    int col = blockIdx.x * blockDim.x + threadIdx.x;  // which column of C
    int row = blockIdx.y * blockDim.y + threadIdx.y;  // which row of C
    if (row < N && col < N) {                          // guard ragged edge
        float acc = 0.0f;
        for (int k = 0; k < N; ++k)
            acc += A[row * N + k] * B[k * N + col];    // dot product of A-row and B-col
        C[row * N + col] = acc;
    }
}

// Naive CPU baseline (triple loop). O(N^3); intentionally simple for a fair "no-GPU" reference.
void matmulCPU(const float* A, const float* B, float* C, int N) {
    for (int i = 0; i < N; ++i)
        for (int j = 0; j < N; ++j) {
            float acc = 0.0f;
            for (int k = 0; k < N; ++k)
                acc += A[i * N + k] * B[k * N + j];
            C[i * N + j] = acc;
        }
}

void fillRandom(float* M, int n) {
    for (int i = 0; i < n; ++i) M[i] = (float)rand() / (float)RAND_MAX;
}

void benchmark(int N) {
    size_t bytes = (size_t)N * N * sizeof(float);
    float *hA = (float*)malloc(bytes), *hB = (float*)malloc(bytes);
    float *hC_cpu = (float*)malloc(bytes), *hC_gpu = (float*)malloc(bytes);
    srand(2708);                          // SEED for reproducible inputs
    fillRandom(hA, N * N); fillRandom(hB, N * N);

    // ---------- CPU baseline ----------
    auto t0 = std::chrono::high_resolution_clock::now();
    matmulCPU(hA, hB, hC_cpu, N);
    auto t1 = std::chrono::high_resolution_clock::now();
    double cpu_ms = std::chrono::duration<double, std::milli>(t1 - t0).count();

    // ---------- GPU ----------
    float *dA, *dB, *dC;
    CUDA_CHECK(cudaMalloc(&dA, bytes));
    CUDA_CHECK(cudaMalloc(&dB, bytes));
    CUDA_CHECK(cudaMalloc(&dC, bytes));

    cudaEvent_t eH2D0, eH2D1, eK0, eK1, eD2H0, eD2H1;
    cudaEventCreate(&eH2D0); cudaEventCreate(&eH2D1);
    cudaEventCreate(&eK0);   cudaEventCreate(&eK1);
    cudaEventCreate(&eD2H0); cudaEventCreate(&eD2H1);

    // Host -> Device
    cudaEventRecord(eH2D0);
    CUDA_CHECK(cudaMemcpy(dA, hA, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(dB, hB, bytes, cudaMemcpyHostToDevice));
    cudaEventRecord(eH2D1);

    // Launch configuration: 2-D grid of 16x16 blocks tiling the N x N output
    dim3 block(BLOCK, BLOCK);
    dim3 grid((N + BLOCK - 1) / BLOCK, (N + BLOCK - 1) / BLOCK);

    // Warm-up launch (excluded from timing) so first-call overhead doesn't skew the kernel time
    matmulKernel<<<grid, block>>>(dA, dB, dC, N);
    CUDA_CHECK(cudaDeviceSynchronize());

    // Timed kernel
    cudaEventRecord(eK0);
    matmulKernel<<<grid, block>>>(dA, dB, dC, N);
    cudaEventRecord(eK1);
    CUDA_CHECK(cudaGetLastError());

    // Device -> Host
    cudaEventRecord(eD2H0);
    CUDA_CHECK(cudaMemcpy(hC_gpu, dC, bytes, cudaMemcpyDeviceToHost));
    cudaEventRecord(eD2H1);
    CUDA_CHECK(cudaEventSynchronize(eD2H1));

    float h2d_ms = 0, kernel_ms = 0, d2h_ms = 0;
    cudaEventElapsedTime(&h2d_ms, eH2D0, eH2D1);
    cudaEventElapsedTime(&kernel_ms, eK0, eK1);
    cudaEventElapsedTime(&d2h_ms, eD2H0, eD2H1);
    float transfer_ms = h2d_ms + d2h_ms;
    float e2e_ms = h2d_ms + kernel_ms + d2h_ms;   // GPU end-to-end (transfers + kernel)

    // ---------- correctness check (max abs diff vs CPU) ----------
    double maxdiff = 0.0;
    for (int i = 0; i < N * N; ++i)
        maxdiff = fmax(maxdiff, fabs((double)hC_cpu[i] - (double)hC_gpu[i]));

    printf("N=%-5d | CPU %10.2f ms | kernel %8.3f ms | H2D+D2H %8.3f ms | "
           "e2e %9.3f ms | speedup(CPU/e2e) %6.2fx | maxdiff %.2e\n",
           N, cpu_ms, kernel_ms, transfer_ms, e2e_ms, cpu_ms / e2e_ms, maxdiff);

    cudaFree(dA); cudaFree(dB); cudaFree(dC);
    free(hA); free(hB); free(hC_cpu); free(hC_gpu);
}

int main() {
    int sizes[] = {256, 1024, 4096};
    printf("Square matmul CPU vs CUDA (BLOCK=%dx%d = %d threads/block)\n",
           BLOCK, BLOCK, BLOCK * BLOCK);
    for (int s : sizes) benchmark(s);
    return 0;
}


Overwriting matmul.cu


### 2. Compile  (`-std=c++14 -arch=$SM`, same flags as the demo)

In [3]:
!nvcc -O3 -std=c++14 -arch=$SM matmul.cu -o matmul

### 3. Run — timing table
Prints, for N = 256, 1024, 4096: CPU ms, GPU kernel ms (cudaEvent), H2D+D2H transfer ms, end-to-end ms, speedup = CPU/end-to-end, and max abs diff vs CPU. **Copy these into `METRICS.md` and `RUN_LOG.txt`.**

In [4]:
!./matmul

Square matmul CPU vs CUDA (BLOCK=16x16 = 256 threads/block)
N=256   | CPU      20.26 ms | kernel    0.103 ms | H2D+D2H    1.035 ms | e2e     1.137 ms | speedup(CPU/e2e)  17.82x | maxdiff 2.29e-05
N=1024  | CPU    3376.59 ms | kernel    9.193 ms | H2D+D2H    5.249 ms | e2e    14.442 ms | speedup(CPU/e2e) 233.81x | maxdiff 9.16e-05
N=4096  | CPU  631476.29 ms | kernel  323.966 ms | H2D+D2H   79.089 ms | e2e   403.055 ms | speedup(CPU/e2e) 1566.73x | maxdiff 4.27e-04


### 4. Profile — separate kernel time from transfer time
State which profiler you used. The Week-1 demo uses **`nvprof`**; try it first. On newer CUDA toolkits `nvprof` is removed — fall back to **Nsight Systems** (`nsys`).

In [5]:
# Primary — nvprof; shows GPU activities: memcpy HtoD/DtoH vs kernel
!nvprof ./matmul 2>&1 | tail -40

==5891== NVPROF is profiling process 5891, command: ./matmul
==5891== Profiling application: ./matmul
Square matmul CPU vs CUDA (BLOCK=16x16 = 256 threads/block)
N=256   | CPU      20.76 ms | kernel    0.113 ms | H2D+D2H    0.416 ms | e2e     0.528 ms | speedup(CPU/e2e)  39.28x | maxdiff 2.29e-05
N=1024  | CPU    3265.82 ms | kernel    9.220 ms | H2D+D2H    4.922 ms | e2e    14.142 ms | speedup(CPU/e2e) 230.93x | maxdiff 9.16e-05
N=4096  | CPU  780330.25 ms | kernel  323.681 ms | H2D+D2H   81.009 ms | e2e   404.690 ms | speedup(CPU/e2e) 1928.22x | maxdiff 4.27e-04
==5891== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   90.06%  742.56ms         6  123.76ms  102.01us  400.34ms  matmulKernel(float const *, float const *, float*, int)
                    5.83%  48.059ms         3  16.020ms  21.951us  46.405ms  [CUDA memcpy DtoH]
                    4.11%  33.909ms         6  5.6515ms  23.615us  18.346ms  [CUDA memcpy H

### 5. Crossover analysis

In these measurements the GPU end-to-end time already beats the CPU at the smallest size tested, N=256 (1.14 ms vs 20.26 ms, a 17.8x speedup), and the margin widens fast with size (233.8x at 1024, 1566.7x at 4096). So the crossover sits below 256 rather than at zero: every GPU call pays fixed costs: kernel launch plus copying 2*N^2 input floats to the device and N^2 back, which for a tiny matrix would swamp the CPU's trivial work. The nvprof breakdown confirms the split: the matmul kernel accounts for ~90% of GPU activity while H2D+D2H transfers are only ~10%, so once N is large enough for the O(N^3) compute to dominate the O(N^2) transfer and launch overhead, the parallel kernel wins decisively; at very small N that overhead dominates and the CPU stays competitive.